# Clasificación de texto con BERT

### Objetivos de Aprendizaje
1. Aprender a cargar un modelo BERT pre-entrenado desde KerasHub
2. Aprender a construir tu propio modelo combinándolo con un clasificador
3. Aprender a entrenar un modelo LLM (BERT) mediante ajuste fino (fine-tuning)
4. Aprender a guardar y usar tu modelo entrenado
5. Aprender a evaluar un modelo de clasificación de texto

Este laboratorio muestra cómo ajustar BERT para realizar análisis de sentimientos en un conjunto de datos de reseñas de películas de IMDB.

## Acerca de BERT

[BERT](https://arxiv.org/abs/1810.04805) (Bidirectional Encoder Representations from Transformers) y otras arquitecturas de codificadores Transformer han tenido un gran éxito en el procesamiento del lenguaje natural (NLP). Calculan representaciones vectoriales del lenguaje natural adecuadas para su uso en modelos de aprendizaje profundo.

Los modelos BERT suelen pre-entrenarse en un gran corpus de texto y luego se ajustan para tareas específicas.

In [ ]:
import os
import warnings
import datetime
import shutil
import keras
import keras_hub as hub
import matplotlib.pyplot as plt
import tensorflow as tf
from google.cloud import aiplatform

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
tf.get_logger().setLevel("ERROR")

### Análisis de Sentimientos

Entrenaremos un modelo para clasificar reseñas de películas como *positivas* o *negativas* usando el [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/) que contiene 50,000 reseñas.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
batch_size = 32
seed = 42

raw_train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=seed,
)
train_ds = raw_train_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Cargando modelos desde Keras Hub

Usaremos una versión llamada "Small BERT", que tiene la misma arquitectura general que el BERT original pero con bloques de Transformer más pequeños.

In [ ]:
bert_encoder = hub.models.BertBackbone.from_preset(
    "bert_small_en_uncased",
    load_weights=True,
)
bert_encoder.summary()

## Modelo de Preprocesamiento

Las entradas de texto deben transformarse en IDs de tokens numéricos antes de entrar a BERT.

In [ ]:
bert_preprocessor = hub.models.BertPreprocessor.from_preset(
    "bert_small_en_uncased", name="preprocessor", truncate="waterfall"
)

### Definición del Modelo

Crearemos un clasificador simple con el preprocesador, el codificador BERT, una capa de Dropout y una capa Dense con activación sigmoide.

In [ ]:
def build_classifier_model(dropout_rate=0.1):
    text_input = keras.layers.Input(shape=(), dtype="string", name="text")
    preprocessor = bert_preprocessor(text_input)
    encoder_outputs = bert_encoder(preprocessor)
    net = encoder_outputs["pooled_output"]
    net = keras.layers.Dropout(dropout_rate, name="dropout")(net)
    net = keras.layers.Dense(1, activation="sigmoid", name="classifier")(net)
    return keras.Model(text_input, net)

classifier_model = build_classifier_model(0.15)

## Entrenamiento del Modelo

Utilizaremos la pérdida `BinaryCrossentropy` y el optimizador `AdamW`.

In [ ]:
loss = keras.losses.BinaryCrossentropy()
metrics = keras.metrics.BinaryAccuracy()
optimizer = keras.optimizers.AdamW(learning_rate=3e-5, weight_decay=0.004)

classifier_model.compile(optimizer=optimizer, loss=loss, metrics=[metrics])

### Exportación para Inferencia

Guardaremos el modelo en formato TensorFlow SavedModel.

In [ ]:
SAVEDMODEL_PATH = "./imdb_bert_savedmodel"
classifier_model.export(SAVEDMODEL_PATH)

## Conclusión

En este laboratorio hemos aprendido a realizar fine-tuning de un modelo BERT para clasificación de texto y cómo preparar el modelo para su despliegue en producción.